In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pylab
from dsautils import cnf; c = cnf.Conf(use_etcd=True); ca = c.get('corr'); ao = ca['antenna_order']
ao

In [ ]:
# get caldata
caldir = '/operations/beamformer_weights/generated/'

caldates = ['0319+415_2025-10-15T09:37:58','2202+422_2025-10-15T04:21:06','0432+416_2025-10-15T10:50:39','0909+428_2025-10-15T15:26:43','1421+417_2025-10-15T20:36:44','0909+428_2025-10-16T15:22:47']
caldata = np.zeros((len(caldates),96, 48*16, 2),dtype=np.complex64)

for i,sb in enumerate(['00','01','02','03','04','05','06','07','08','09','10','11','12','13','14','15']):
    for j,caldate in enumerate(caldates):
        with open(caldir+'beamformer_weights_sb'+sb+'_'+caldate+'.dat', 'rb') as f:
            data = np.fromfile(f, '<f4')
            gains = data[192:].reshape(96, 48, 2, 2)
            gains1 = gains[..., 0]+1.0j*gains[..., 1]
            caldata[j,:,i*48:(i+1)*48,:] = gains1
            if j==0:
                g10 = gains1
            caldata[j,:,i*48:(i+1)*48,:] /= g10
        

In [ ]:
# plot for all antennas
plt.figure(figsize=(16,40))
freqs = 1530.-1024.*250./6144.-np.arange(768)*250./1024.
plt.subplots_adjust(wspace=0.3,hspace=0.35)

offset=0
for i in np.arange(96):
    
    pylab.subplot(16,6,i+1)
    plt.xlim(1280.,1530.)
    plt.ylim(-50.,50.)
    for j in np.arange(len(caldates)):
        plt.plot(freqs,(180./np.pi)*(np.angle(caldata[j,i+offset,:,0])),label=caldates[j])
        #plt.plot(freqs,(180./np.pi)*np.angle(caldata[j,i,:,1]))
    if i==0:
        plt.legend()
    plt.title('Antenna '+str(ao[str(i+offset)]))
    
plt.savefig("fig.pdf",bbox_inches="tight")
    